In [1]:
# importing libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from lightgbm import LGBMClassifier

train_df = pd.read_pickle("../data/train.pkl")

In [2]:
from sklearn.metrics import (
    accuracy_score ,
    precision_score ,
    recall_score ,
    f1_score ,
    average_precision_score , 
    roc_auc_score ,
    confusion_matrix ,
    classification_report, 
    ConfusionMatrixDisplay 
)

In [3]:
train_df = train_df.sort_values(["Transaction_Day" , "Transaction_Time_Of_Day"]).reset_index(drop=True)

In [4]:
x = train_df.drop(columns=["isFraud" , "TransactionID" ,"TransactionDT"])
y = train_df["isFraud"]

In [5]:
# NUMERICAL

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median' , add_indicator=True)) ,
    ('scaler' , StandardScaler())
])

In [6]:
# CATEGORICAL

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant' , fill_value="Missing")) ,
    ('encoder' , OneHotEncoder(handle_unknown='ignore'))
])

In [7]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5 , gap=0)

In [8]:
seeds = [123 , 2024 , 777]
lgbmc_seed_results = {}

for seed in seeds:
    print(f"\n{'='*60}")
    print(f"LightGBM - SEED-{seed}")
    print(f"{'='*60}")

    oof_predictions = np.full(len(x) , np.nan)

    for fold , (train_idx , val_idx) in enumerate(tscv.split(x) , start=1):
        x_train = x.iloc[train_idx].copy()
        x_val = x.iloc[val_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_val = y.iloc[val_idx].copy()

        numeric_cols = x_train.select_dtypes(include=["int64" , "float64" , "uint16" , "uint8"]).columns
        categorical_cols = x_train.select_dtypes(include=["object"]).columns

        # Combined Pipeline
        preprocessor = ColumnTransformer([
            ('num' , numeric_pipeline , numeric_cols) ,
            ('cat' , categorical_pipeline , categorical_cols)
        ])

        lgbmc_pipeline = Pipeline([
            ('preprocessor' , preprocessor) ,
            ('model' , LGBMClassifier( objective="binary" , random_state=seed , n_jobs=-1 , max_depth=-1 , n_estimators=100 , learning_rate = 0.05 ,subsample=0.8 ,min_child_weight = 5, 
                    num_leaves=31 , colsample_bytree = 0.8, verbosity=-1 , subsample_freq=1,  min_child_samples=20))
        ])

        lgbmc_pipeline.fit(x_train , y_train)
        val_pred = lgbmc_pipeline.predict(x_val)
        val_prob = lgbmc_pipeline.predict_proba(x_val)[:,1]
        oof_predictions[val_idx] = val_prob

        roc_auc = roc_auc_score(y_val , val_prob)
        pr_auc = average_precision_score(y_val , val_prob)
        recall = recall_score(y_val , val_pred)
        f1 = f1_score(y_val , val_pred)
        precision = precision_score(y_val , val_pred)


        print(
            f"Fold {fold} | "
            f"ROC-AUC : {roc_auc:.4f} | " 
            f"PR-AUC : {pr_auc:.4f} | "
            f"Recall = {recall:.4f} | "
            f"F1 Score = {f1:.4f} | "
            f"Precision = {precision:.4f}")
        print(confusion_matrix(y_val , val_pred))

    oof_mask = ~np.isnan(oof_predictions)
    overall_roc_auc = roc_auc_score(y.iloc[oof_mask] , oof_predictions[oof_mask])
    overall_pr_auc = average_precision_score(y.iloc[oof_mask] , oof_predictions[oof_mask])
    print(f"OOF ROC-AUC : {overall_roc_auc:.4f}")
    print(f"OOF PR-AUC : {overall_pr_auc:.4f}")
    lgbmc_seed_results[seed] = {
        "oof" : oof_predictions ,
        "roc-auc" : overall_roc_auc , 
        "pr-auc" : overall_pr_auc
    }


LightGBM - SEED-123


Fold 1 | ROC-AUC : 0.8903 | PR-AUC : 0.5457 | Recall = 0.2924 | F1 Score = 0.4415 | Precision = 0.9004
[[95009   107]
 [ 2340   967]]
Fold 2 | ROC-AUC : 0.8983 | PR-AUC : 0.5765 | Recall = 0.3704 | F1 Score = 0.5156 | Precision = 0.8482
[[94096   269]
 [ 2555  1503]]
Fold 3 | ROC-AUC : 0.8907 | PR-AUC : 0.5265 | Recall = 0.2954 | F1 Score = 0.4414 | Precision = 0.8726
[[94578   159]
 [ 2597  1089]]
Fold 4 | ROC-AUC : 0.8979 | PR-AUC : 0.5569 | Recall = 0.3562 | F1 Score = 0.4978 | Precision = 0.8263
[[94461   276]
 [ 2373  1313]]
Fold 5 | ROC-AUC : 0.8994 | PR-AUC : 0.5267 | Recall = 0.3326 | F1 Score = 0.4704 | Precision = 0.8031
[[94749   277]
 [ 2267  1130]]
OOF ROC-AUC : 0.8937
OOF PR-AUC : 0.5440

LightGBM - SEED-2024
Fold 1 | ROC-AUC : 0.8909 | PR-AUC : 0.5455 | Recall = 0.2927 | F1 Score = 0.4423 | Precision = 0.9047
[[95014   102]
 [ 2339   968]]
Fold 2 | ROC-AUC : 0.8963 | PR-AUC : 0.5804 | Recall = 0.3679 | F1 Score = 0.5166 | Precision = 0.8670
[[94136   229]
 [ 2565  1493]]

In [11]:
lgbmc_seed_results[123]

{'oof': array([       nan,        nan,        nan, ..., 0.0047369 , 0.06364743,
        0.01164506], shape=(590540,)),
 'roc-auc': 0.8936911004753996,
 'pr-auc': 0.54402702625757}

In [12]:
for seed in seeds:
    pd.DataFrame({
        "isFraud" : y.values , 
        f"lightbgm_seed_{seed}_oof" : lgbmc_seed_results[seed]["oof"]
    }).to_csv(f"../outputs/oof/lightgbm_seed_{seed}_oof.csv" , index=False)